In [3]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import torch.optim as optim
from sympy import symbols, lambdify
from sympy import symbols, hermite
from torch.distributions import Normal
from numpy.polynomial.hermite import Hermite, herm2poly

In [4]:
num_train_functions = 1000  
train_iterations    = 10000  
test_functions      = 10  
num_train_functions = 10  
train_iterations    = 1000  
test_functions      = 3  


In [5]:
# ------------------------
# Snippet 0: Device Setup
# ------------------------
# Use Apple M1’s GPU via MPS if available, else CPU
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"Using device: {device}\n")

Using device: mps



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------------------------------------
# 1) Hermite polynomials H0…H3 (physicists’ version)
# -----------------------------------------------------------
x_sym = symbols('x')

def sample_hermite_function(p, q, coeff_variance=1.0):
    """
    Returns a NumPy‐vectorized function f(x) = sum_{i=p}^q c_i H_i(x),
    with c_i ~ N(0, coeff_variance).
    """
    expr = 0
    for i in range(p, q+1):
        c_i = np.random.normal(scale=np.sqrt(coeff_variance))
        expr += c_i * hermite(i, x_sym)
    return lambdify(x_sym, expr, 'numpy')

In [ ]:
# -----------------------------------------------------------
# 3) Single‐head causal self‐attention module
# -----------------------------------------------------------
class SingleHeadCausalSelfAttention(nn.Module):
    def __init__(self, input_dim=2, d_att=8):
        """
        input_dim: 2  (since each context row is [x_i, y_i])
        d_att:      size of the attention embedding/output (must be > 2)
        """
        super().__init__()
        self.d_att = d_att
        # We learn three linear maps from input_dim→d_att, no bias
        self.W_Q = nn.Linear(input_dim, d_att, bias=False)
        self.W_K = nn.Linear(input_dim, d_att, bias=False)
        self.W_V = nn.Linear(input_dim, d_att, bias=False)

    def forward(self, context):
        """
        context: tensor of shape (n, 2)  – each row is [x_i, y_i]
        Returns:
          attn_out : shape (n, d_att)
        """
        n = context.size(0)  # sequence length
        # Compute Q, K, V
        Q = self.W_Q(context)  # (n, d_att)
        K = self.W_K(context)  # (n, d_att)
        V = self.W_V(context)  # (n, d_att)

        # Raw scores: (n, n) = Q @ K^T / sqrt(d_att)
        scores = (Q @ K.transpose(0, 1)) / np.sqrt(self.d_att)

        # Causal mask: positions j > i should be −∞
        # torch.triu(torch.ones(n,n), diagonal=1) == 1 for entries above diagonal.
        causal_mask = torch.triu(torch.ones(n, n, device=context.device) * float("-inf"), diagonal=1)
        scores = scores + causal_mask  # broadcasting

        attn_weights = F.softmax(scores, dim=-1)  # (n, n)
        attn_out = attn_weights @ V               # (n, d_att)
        return attn_out

# -----------------------------------------------------------
# 4) Full “ContextPredictor” model = Attention → Pool → MLP
# -----------------------------------------------------------
class ContextPredictor(nn.Module):
    def __init__(self, d_att=8, d_proj=16, hidden_dim=32):
        """
        d_att:      the attention embedding size
        d_proj:     the dimension after projecting the aggregated attention
        hidden_dim: hidden‐layer size in the final MLP
        """
        super().__init__()
        # (a) single‐head causal self‐attention
        self.attention = SingleHeadCausalSelfAttention(input_dim=2, d_att=d_att)
        # (b) a projection layer to map the averaged attention output → d_proj
        self.proj_layer = nn.Linear(d_att, d_proj)
        # (c) one hidden layer + ReLU + output to a single scalar
        self.hidden_layer = nn.Linear(d_proj + 1, hidden_dim)
        self.output_layer = nn.Linear(hidden_dim, 1)

    def forward(self, context_x, context_y, x_test):
        """
        context_x: shape (n,)
        context_y: shape (n,)
        x_test:    shape (1,)   (scalar as a 1‐element tensor)
        Returns:
          y_pred: shape (1,)   (predicted scalar)
        """
        # (1) stack context into shape (n, 2)
        context = torch.stack([context_x, context_y], dim=1)  # (n, 2)

        # (2) apply self‐attention → (n, d_att)
        attn_out = self.attention(context)

        # (3) aggregate: simply average over sequence length → (d_att,)
        agg = attn_out.mean(dim=0)

        # (4) project to d_proj → (d_proj,)
        proj_vec = self.proj_layer(agg)

        # (5) concatenate with x_test scalar → (d_proj + 1,)
        inp = torch.cat([proj_vec, x_test], dim=0)

        # (6) MLP: hidden + ReLU + output
        h = F.relu(self.hidden_layer(inp))
        y_pred = self.output_layer(h)  # (1,)

        return y_pred

# -----------------------------------------------------------
# 5) Training + Plotting + Testing routine
# -----------------------------------------------------------
def train_and_evaluate(
    num_train_functions=10,
    contexts_per_function=5,
    train_iterations=1000,
    test_functions=3,
    device="cpu"
):
    """
    1) Sample `num_train_functions` random Hermite‐functions.
    2) For each function, sample `contexts_per_function` contexts.
       Each context has n∈[50,100] pts drawn x∼N(0,1), y=f(x), plus one test point.
    3) For each function, we run a total of `train_iterations` updates. On each update,
       we randomly pick one of that function’s contexts, compute MSE, and step Adam.
    4) After finishing training all `num_train_functions`, we:
       • plot the last function’s last context (scatter context, true test point, predicted test point)
       • test on `test_functions` new Hermite‐functions and report |y_pred – y_true|.
    """
    # Move model & data to device
    model = ContextPredictor(d_att=8, d_proj=16, hidden_dim=32).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    # Will hold (context_x, context_y, x_test, y_test) for the “last” function & context
    last_context = None
    last_test_point = None
    last_prediction = None

    # ───────────────────────────────────────────────────────────
    # (A) TRAINING PHASE
    # ───────────────────────────────────────────────────────────
    for func_idx in range(num_train_functions):
        f = sample_hermite_function(device=device)

        # Pre‐sample all contexts for this function, so we can pick them at random
        contexts = []
        for ctx_idx in range(contexts_per_function):
            n = np.random.randint(50, 101)
            context_x = torch.randn(n, device=device)
            context_y = f(context_x)
            x_test = torch.randn(1, device=device)
            y_test = f(x_test)
            contexts.append((context_x, context_y, x_test, y_test))

        # Run exactly `train_iterations` gradient updates for this function
        for it in range(train_iterations):
            model.train()
            # Pick one of the contexts uniformly at random
            context_x, context_y, x_test, y_test = contexts[np.random.randint(contexts_per_function)]
            optimizer.zero_grad()
            y_pred = model(context_x, context_y, x_test)  # returns shape (1,)
            loss = F.mse_loss(y_pred.reshape(()), y_test.reshape(()))
            loss.backward()
            optimizer.step()

        # If this is the very last function, store its last context for plotting:
        if func_idx == num_train_functions - 1:
            # Use the “last” context in that list
            context_x, context_y, x_test, y_test = contexts[-1]
            model.eval()
            with torch.no_grad():
                y_pred_val = model(context_x, context_y, x_test).item()
            last_context = (context_x.cpu(), context_y.cpu())
            last_test_point = (x_test.item(), y_test.item())
            last_prediction = y_pred_val

    # ───────────────────────────────────────────────────────────
    # (B) PLOTTING the “LAST training instance”
    # ───────────────────────────────────────────────────────────
    ctx_x, ctx_y = last_context
    tx, ty = last_test_point  # true test point
    yp = last_prediction      # predicted

    plt.figure(figsize=(6, 4))
    plt.scatter(ctx_x.numpy(), ctx_y.numpy(), label="Context Points", alpha=0.6)
    plt.scatter([tx], [ty], color="green", marker="*", s=100, label="True Test Point")
    plt.scatter([tx], [yp], color="red", marker="x", s=100, label="Predicted Test Point")
    plt.title("Last Training Instance (Function #{})".format(num_train_functions))
    plt.xlabel("x")
    plt.ylabel("y")
    plt.legend()
    plt.show()

    # ───────────────────────────────────────────────────────────
    # (C) TESTING PHASE on NEW FUNCTIONS
    # ───────────────────────────────────────────────────────────
    errors = []
    for test_idx in range(test_functions):
        f_test = sample_hermite_function(device=device)
        n = np.random.randint(50, 101)
        context_x = torch.randn(n, device=device)
        context_y = f_test(context_x)
        x_test = torch.randn(1, device=device)
        y_true = f_test(x_test).item()

        model.eval()
        with torch.no_grad():
            y_pred = model(context_x, context_y, x_test).item()

        abs_err = abs(y_pred - y_true)
        errors.append(abs_err)
        print(f"Test Function {test_idx+1:2d}  → True y = {y_true: .4f},  Predicted y = {y_pred: .4f},  |Error| = {abs_err: .4f}")

    print(f"\nMean absolute error over {test_functions} test functions: {np.mean(errors):.4f}")

# ───────────────────────────────────────────────────────────────
# To run a quick sanity test, use small numbers:
# ─────────────────────────────────────────────────────────────────
train_and_evaluate(
    num_train_functions=10,    # ← set to 1000 for full run
    contexts_per_function=5,   # exactly as requested
    train_iterations=1000,     # ← set to 10000 for full run
    test_functions=3,          # ← set to 10 for full run
    device="cpu"
)
